# Analytics for Value Investing - Group Project
## S&P 500 Ex-Financials Quantitative Backtest (2010–2025)

**Team Members:**

<div style="display: inline-block">

| Name | Student ID | Main Role |
| :-------- | :-------- | :-------- |
| Kathryn Chan Hui | 01532587 | Development |
| Ng Hock Wah | 01513633 | Development |
| Liu Yi-Yi | 01548034 | Report |
| Liu Zhemin | 01525938 | Report |
| Zhang Tianrui | 01526062 | Slides |
| Ma Sunan | 01528539 | Slides |

</div>

**Course:** ACCT 656 - Analytics for Value Investing<br>
**Submission Date:** 19 June 2026

### Project Objective
To develop an investment strategy mimicking Warren Buffett’s value investing style using the Frazzini et al. (2018) methodology. 

### Rules & Constraints
* **Universe:** S&P 500 companies, strictly excluding all banks and financial institutions.
* **Time Horizon:** 16 years of data, ending no earlier than 2025 (2010–2025).
* **Data Frequency:** Strictly limited to monthly stock price/return data and annual financial/accounting data.

In [1]:
## IMPORT LIBRARIES

# General libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import time
import warnings
# Financial libraries
import yfinance as yf

# Suppress warnings for cleaner submission output
warnings.filterwarnings('ignore')
# Set plotting aesthetics for report-ready charts
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## LOAD DATA

print("Loading project datasets...")
snp_raw = pd.read_csv("data/compustat_fundamental_annual.csv") # raw data from Compustat
sp500 = pd.read_csv("data/constituents.csv") # data of S&P 500 constituents

Loading project datasets...


### 1. Data preparation
Before conducting financial ratio analysis, the raw Compustat data has to be processed.
1. `costat` column was removed as there is only 1 unique value
2. Entries of `USD` currency (`curcd`) were kept, `CAD` entries were removed to prevent FX conversion issues and restrict the universe to the U.S. metrics
3. `datafmt`, `indfmt`, `consol` columns were removed as they are redundant database identifiers
4. `fyr` column was removed as we can calculate the fiscal year-end date using `datadate` column

In [2]:
# snp_raw["sic"].value_counts()

# --------------------------------------------------
# 1.1 Clean up raw Compustat data
# --------------------------------------------------

# Drop costat column since it is not relevant to our analysis
snp_raw.drop("costat", inplace=True, axis=1)
# Drop CAD since we dont want to convert it to USD
snp_raw.drop(snp_raw[snp_raw["curcd"] == "CAD"].index, inplace=True, axis=0)
# Drop curcd column since we only have USD data
snp_raw.drop("curcd", inplace=True, axis=1)
# Drop datafmt, indfmt and consol columns since they are not relevant to our analysis
snp_raw.drop(["datafmt", "indfmt", "consol"], inplace=True, axis=1)
# Drop fyr column since we can calculate fiscal year end date using datadate column
snp_raw.drop("fyr", inplace=True, axis=1)
snp_raw.head()

,gvkey,datadate,conm,tic,cik,naics,sic,at,ceq,che,...,ebit,ebitda,gp,ni,revt,sale,capx,oancf,csho,prcc_f
0,1004,31/5/2009,AAR CORP,AIR,1750.0,423860.0,5080,1377.511,656.895,112.505,...,125.529,166.080,313.299,78.651,1423.976,1423.976,27.535,64.451,38.884,14.70
1,1004,31/5/2010,AAR CORP,AIR,1750.0,423860.0,5080,1501.042,746.906,79.370,...,95.415,134.345,286.249,44.628,1352.151,1352.151,28.855,153.156,39.484,19.70
2,1004,31/5/2011,AAR CORP,AIR,1750.0,423860.0,5080,1703.727,835.845,57.433,...,137.016,196.312,367.711,69.826,1775.782,1775.782,124.879,108.598,39.781,26.39
3,1004,31/5/2012,AAR CORP,AIR,1750.0,423860.0,5080,2195.653,864.649,67.720,...,142.360,222.693,412.090,67.723,2074.498,2074.498,91.218,94.217,40.273,12.05
4,1004,31/5/2013,AAR CORP,AIR,1750.0,423860.0,5080,2136.900,918.600,75.300,...,136.600,245.200,452.600,55.000,2167.100,2167.100,37.600,162.900,39.382,20.06


As bank and financial institutions have different financial statement structure, we will drop them from our analysis. This is done by analysing the data of S&P 500 constituents to exclude GIC Sectors that are of value `Financials`. Following this, any banking-related companies using sub-industry keywords are removed.

It is noted that the number of companies reduced from 503 to 427 companies.

In [3]:
sp500.head() # check the first few rows

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [4]:
# --------------------------------------------------
# 1.2 Remove companies under Financials sector
# --------------------------------------------------

sp500_nonfin = sp500[
    sp500["GICS Sector"] != "Financials"
].copy()

print("Original number of S&P 500 companies:", len(sp500))
print("After removing Financials:", len(sp500_nonfin))

Original number of S&P 500 companies: 503
After removing Financials: 427


In [5]:
# --------------------------------------------------
# 1.2 Remove banking-related companies using sub-industry keywords
# --------------------------------------------------

banking_keywords = [
    "bank",
    "banks",
    "banking",
    "financial",
    "insurance",
    "capital markets",
    "mortgage",
    "consumer finance",
    "diversified financial",
    "asset management",
    "investment banking",
    "brokerage"
]

pattern = "|".join(banking_keywords)
sp500_nonfin = sp500_nonfin[
    ~sp500_nonfin["GICS Sub-Industry"]
    .str.lower()
    .str.contains(pattern, na=False)
].copy()

print("After removing Financials and banking-related firms:", len(sp500_nonfin))

After removing Financials and banking-related firms: 427


We now create a Yahoo Finance-compatible ticker that cleans the symbol provided to create a standardised ticker that will be used to filter the data in `snp_raw`.

In [6]:
# --------------------------------------------------
# 1.3 Create Yahoo Finance-compatible ticker
# --------------------------------------------------

sp500_nonfin["ticker_yahoo"] = (
    sp500_nonfin["Symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(".", "-", regex=False)
)

sp500_nonfin[["Symbol", "ticker_yahoo", "Security", "GICS Sector", "GICS Sub-Industry"]].head()

,Symbol,ticker_yahoo,Security,GICS Sector,GICS Sub-Industry
0,MMM,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,AOS,A. O. Smith,Industrials,Building Products
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services


Non-Financial tickers are now merged with `snp_raw` to keep only non-financial company entries

In [7]:
# --------------------------------------------------
# 1.4 Merge S&P 500 data with Yahoo Finance-compatible tickers
# --------------------------------------------------

snp500_overall = pd.merge(snp_raw, sp500_nonfin, left_on="tic", right_on="ticker_yahoo", how="inner")
snp500_overall.drop(["tic", "Symbol"], inplace=True, axis=1)
snp500_overall.head()

,gvkey,datadate,conm,cik,naics,sic,at,ceq,che,dlc,...,csho,prcc_f,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,ticker_yahoo
0,1075,31/12/2009,PINNACLE WEST CAPITAL CORP,764622.0,2211.0,4911,11808.155,3316.109,145.378,431.408,...,101.435,36.58,Pinnacle West Capital,Utilities,Multi-Utilities,"Phoenix, Arizona",1999-10-04,764622,1985,PNW
1,1075,31/12/2010,PINNACLE WEST CAPITAL CORP,764622.0,2211.0,4911,12362.703,3683.327,110.188,648.479,...,108.770,41.45,Pinnacle West Capital,Utilities,Multi-Utilities,"Phoenix, Arizona",1999-10-04,764622,1985,PNW
2,1075,31/12/2011,PINNACLE WEST CAPITAL CORP,764622.0,2211.0,4911,13111.018,3821.850,33.583,477.435,...,109.246,48.18,Pinnacle West Capital,Utilities,Multi-Utilities,"Phoenix, Arizona",1999-10-04,764622,1985,PNW
3,1075,31/12/2012,PINNACLE WEST CAPITAL CORP,764622.0,2211.0,4911,13379.615,3972.806,26.202,215.003,...,109.743,50.98,Pinnacle West Capital,Utilities,Multi-Utilities,"Phoenix, Arizona",1999-10-04,764622,1985,PNW
4,1075,31/12/2013,PINNACLE WEST CAPITAL CORP,764622.0,2211.0,4911,13508.686,4194.470,9.526,693.549,...,110.182,52.92,Pinnacle West Capital,Utilities,Multi-Utilities,"Phoenix, Arizona",1999-10-04,764622,1985,PNW


In [8]:
# Store merged data to understand the data content
snp500_overall.to_csv("data/snp500_merged.csv", index=False)

### 2. Data cleaning
Before working with the data to calculate financial ratios, a quick review of the contents of the S&P 500 raw data from Compustat showed some entries having no financial data. If more than 25% of a row is missing, the firm-year observation is likely too incomplete to be reliable. To confirm the removal, this is done on a small sample of data where known entries are to be omitted.

After confirming this is the threshold to be used, the removal will be done on the whole dataset.

In [9]:
# --------------------------------------------------
# 2.1 Testing removal of entries with more than 25% missing values
# --------------------------------------------------

# Checking specific rows to confirm missing values are handled correctly
copy = snp500_overall.iloc[6840:6845].copy()
## we want to remove rows with more than 25% missing values, so we check rows around the 25% threshold
cleaned = copy[copy.isna().mean(axis=1) <= 0.25]
# Printed copy and cleaned to confirm that the correct rows are removed

In [10]:
# --------------------------------------------------
# 2.2 Conduct actual removal of entries with more than 25% missing values
# --------------------------------------------------

snp500_overall_cleaned = snp500_overall[snp500_overall.isna().mean(axis=1) <= 0.25]
print("Original number of rows:", len(snp500_overall))
print("Number of rows after cleaning:", len(snp500_overall_cleaned))
print("Number of rows removed:", len(snp500_overall) - len(snp500_overall_cleaned))

Original number of rows: 6878
Number of rows after cleaning: 6846
Number of rows removed: 32


Note: cross checked with excel, all 32 rows of empty data was removed

In [11]:
# Store cleaned data for analysis
snp500_overall_cleaned.to_csv("data/snp500_cleaned.csv", index=False)

### 3. Downloading Yahoo monthly stock returns
To evaluate performance and calculate market-based signals, monthly pricing data is required. The `yfinance` API is used to download monthly adjusted closing prices from 2009 to 2025 for the non-financial S&P 500 universe - established in Section 2.

*Note: The download deliberately begins in January 2009 to ensure we have a full trailing year of data to calculate lagged market signals (like 11-month Momentum) by the time the backtest officially begins in 2010. Any failed downloads (e.g., delisted tickers like FDXF) would be tracked and isolated for removal.*

In [12]:
# Load the cleaned S&P 500 data for analysis

snp500_cleaned = pd.read_csv("data/snp500_cleaned.csv")

tickers = (
    snp500_cleaned["ticker_yahoo"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
    .tolist()
)

tickers = sorted(tickers)
print("Number of tickers:", len(tickers))

Number of tickers: 423


To calculate monthly stock returns, the filtered universe is iterated through to download historical pricing data using the `yfinance` API. The adjusted closing price is extracted with `auto_adjust=True`. This ensures that subsequent return calculations (using `pct_change()`) is accounted for. Any failed downloads due to delisted tickers are logged and excluded.

In [13]:
# --------------------------------------------------
# 3.1 Download monthly adjusted prices from Yahoo Finance
# --------------------------------------------------

all_monthly_data = []
failed_tickers = []

for i, ticker in enumerate(tickers, start=1):
    try:
        print(f"Downloading {i}/{len(tickers)}: {ticker}")

        data = yf.download(
            ticker,
            start="2009-01-01",
            end="2025-12-31",
            interval="1mo",
            auto_adjust=True,
            progress=False,
            multi_level_index=False
        )

        if data.empty:
            print(f"No data found for: {ticker}")
            failed_tickers.append(ticker)
            continue

        # Reset index so Date becomes a column
        data = data.reset_index()
        # Keep only date and adjusted close price
        temp = data[["Date", "Close"]].copy()

        temp["ticker_yahoo"] = ticker
        temp = temp.rename(columns={
            "Date": "month",
            "Close": "adjusted_close"
        })

        temp["month"] = pd.to_datetime(temp["month"])
        # Calculate monthly return ticker by ticker
        temp["monthly_return"] = temp["adjusted_close"].pct_change()

        all_monthly_data.append(
            temp[["ticker_yahoo", "month", "adjusted_close", "monthly_return"]]
        )

        time.sleep(0.3)

    except Exception as e:
        print(f"Failed to download {ticker}: {e}")
        failed_tickers.append(ticker)

$FDXF: possibly delisted; no price data found  (1mo 2009-01-01 -> 2025-12-31) (Yahoo error = "Data doesn't exist for startDate = 1230786000, endDate = 1767157200")

1 Failed download:
['FDXF']: possibly delisted; no price data found  (1mo 2009-01-01 -> 2025-12-31) (Yahoo error = "Data doesn't exist for startDate = 1230786000, endDate = 1767157200")


No data found for: FDXF


The monthly stock returns data are now concatenatedinto a single consolidated dataframe and sorted chronologically by company. To maintain data integrity for the remainder of the backtest, the cleaned monthly stock data and the list of failed downloads (e.g.: delisted tickers) into separate CSV files for further analysis and cross-referencing.

In [14]:
# --------------------------------------------------
# 3.2 Combine all downloaded monthly data
# --------------------------------------------------

monthly_stock_data = pd.concat(all_monthly_data, ignore_index=True)

monthly_stock_data = monthly_stock_data.sort_values(
    ["ticker_yahoo", "month"]
).reset_index(drop=True)

# Save clean monthly return file
monthly_stock_data.to_csv("data/yahoo_monthly_stock_returns_clean.csv", index=False)

# Save failed tickers
failed_df = pd.DataFrame({"failed_ticker": failed_tickers})
failed_df.to_csv("data/failed_yahoo_tickers.csv", index=False)

print("Monthly stock return file saved as: data/yahoo_monthly_stock_returns_clean.csv")
print("Failed ticker file saved as: data/failed_yahoo_tickers.csv")
print("Number of failed tickers:", len(failed_tickers))

Monthly stock return file saved as: data/yahoo_monthly_stock_returns_clean.csv
Failed ticker file saved as: data/failed_yahoo_tickers.csv
Number of failed tickers: 1


To ensure the integrity of the downloaded Yahoo Finance pricing data before constructing signals, a final review of the consolidated data is conducted. The following metrics are displayed:
- **Shape of data**: To note the total number of monthly-firm observations
- **Date range**: To verify the timeframe correctly spans the specified 2009 - 2025 horizon
- **Columns**: To verify the presence of required variables (`adjusted_close` and `monthly_return`) for upcoming analysis
- **Sample**: A preview of the first 20 chronological observations

In [15]:
# --------------------------------------------------
# 3.3 Validation checks
# --------------------------------------------------

print("Shape:", monthly_stock_data.shape)

print(f"\nDate range: {monthly_stock_data['month'].min()} to {monthly_stock_data['month'].max()}")
print(f"Number of tickers downloaded: {monthly_stock_data['ticker_yahoo'].nunique()}")
print(f"Columns: {monthly_stock_data.columns.tolist()}")

print("\nSample:")
monthly_stock_data.head(20)

Shape: (80285, 4)

Date range: 2009-01-01 00:00:00 to 2025-12-01 00:00:00
Number of tickers downloaded: 422
Columns: ['ticker_yahoo', 'month', 'adjusted_close', 'monthly_return']

Sample:


,ticker_yahoo,month,adjusted_close,monthly_return
0,A,2009-01-01,11.443534,NaN
1,A,2009-02-01,8.778863,-0.232854
2,A,2009-03-01,9.728271,0.108147
3,A,2009-04-01,11.557460,0.188028
4,A,2009-05-01,11.538472,-0.001643
5,A,2009-06-01,12.854987,0.114098
6,A,2009-07-01,14.696836,0.143279
7,A,2009-08-01,16.253860,0.105943
8,A,2009-09-01,17.614683,0.083723
9,A,2009-10-01,15.658914,-0.111031


### 4. Aligning S&P 500 Universe (Removing failed tickers)
Before constructing financial signals, consistency should be established between the annual accounting data (Compustat) and the available monthly stock closing pricing data (Yahoo Finance).

By using the `failed_yahoo_tickers.csv` log, we can identify companies that lack pricing data (e.g.: delisted ticker `FDXF`) and remove their corresponding firm-year observations from the Compustat data.

The final universe has 422 tradeable S&P 500 companies.

In [16]:
# --------------------------------------------------
# 4.1 Load cleaned Compustat + S&P 500 file
# --------------------------------------------------

snp500_cleaned = pd.read_csv("data/snp500_cleaned.csv")

# --------------------------------------------------
# 4.2 Load failed Yahoo ticker file
# --------------------------------------------------

failed = pd.read_csv("data/failed_yahoo_tickers.csv")

failed_tickers = (
    failed["failed_ticker"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .tolist()
)

print("Failed tickers:", failed_tickers)
print("Number of failed tickers:", len(failed_tickers))

# --------------------------------------------------
# 4.3 Remove failed tickers from Compustat/S&P 500 file
# --------------------------------------------------

snp500_cleaned2 = snp500_cleaned[
    ~snp500_cleaned["ticker_yahoo"].isin(failed_tickers)
].copy()

print("\nOriginal rows:", len(snp500_cleaned))
print("Rows after removing failed tickers:", len(snp500_cleaned2))
print("Rows removed:", len(snp500_cleaned) - len(snp500_cleaned2))
print("Original unique tickers:", snp500_cleaned["ticker_yahoo"].nunique())
print("Cleaned unique tickers:", snp500_cleaned2["ticker_yahoo"].nunique())

# Save final cleaned file
snp500_cleaned2.to_csv("data/snp500_cleaned_final.csv", index=False)
print("\nFinal cleaned file saved as: data/snp500_cleaned_final.csv")

Failed tickers: ['FDXF']
Number of failed tickers: 1

Original rows: 6846
Rows after removing failed tickers: 6844
Rows removed: 2
Original unique tickers: 423
Cleaned unique tickers: 422

Final cleaned file saved as: data/snp500_cleaned_final.csv


A set-based consistency check is done to mathematically prove that the accounting and pricing datasets contain the exact same universe of 422 tradeable S&P 500 companies.

In [17]:
# --------------------------------------------------
# 4.4 Check ticker consistency between accounting and return files
# --------------------------------------------------

accounting_tickers = set(snp500_cleaned2["ticker_yahoo"].dropna().unique())
return_tickers = set(monthly_stock_data["ticker_yahoo"].dropna().unique())

print("Tickers in accounting file:", len(accounting_tickers))
print("Tickers in monthly return file:", len(return_tickers))

print("Accounting tickers not in return file:")
print(accounting_tickers - return_tickers)

print("Return tickers not in accounting file:")
print(return_tickers - accounting_tickers)

Tickers in accounting file: 422
Tickers in monthly return file: 422
Accounting tickers not in return file:
set()
Return tickers not in accounting file:
set()


### 5. Final Accounting Data Preparation
Before creating the financial accounting signals, the raw Compustat data has to be properly formatted.

All columns are standardised, and rigorous coercion is done on essential accounting metrics into numeric values (using `pd.numeric()`) to prevent data type errors during calculation. A unified fiscal year (`fyear`) is created and sorted chronologically by company to prepare for the lagged factor calculations.

In [18]:
# ------------------------------------------------------------
# 5.1 Load final cleaned Compustat + S&P 500 data
# ------------------------------------------------------------

snp500 = pd.read_csv("data/snp500_cleaned_final.csv")

# Standardize column names
snp500.columns = snp500.columns.str.lower()
# Remove duplicate columns if any, e.g. duplicated cik
snp500 = snp500.loc[:, ~snp500.columns.duplicated()].copy()

# Convert datadate to datetime
snp500["datadate"] = pd.to_datetime(snp500["datadate"], errors="coerce")

# Create fiscal year if fyear is not available
if "fyear" not in snp500.columns:
    snp500["fyear"] = snp500["datadate"].dt.year

# ------------------------------------------------------------
# 5.2 Convert required columns to numeric
# ------------------------------------------------------------

numeric_cols = [
    "fyear", "at", "ceq", "seq", "txditc",
    "pstk", "pstkl", "pstkrv",
    "ni", "gp", "dlc", "dltt",
    "prcc_f", "csho"
]

for col in numeric_cols:
    if col in snp500.columns:
        snp500[col] = pd.to_numeric(snp500[col], errors="coerce")

# ------------------------------------------------------------
# 5.3 Sort data by company and fiscal year
# ------------------------------------------------------------

snp500 = snp500.sort_values(["ticker_yahoo", "fyear"]).copy()

### 6. Generating Financial Signals
Fundamental accounting metrics and factor signals are constructed to mimic a Buffett-style value and quality strategy, adapting Frazzini's methodology.

**6.1 Construct Book Equity**<br>
- Shareholders' Equity + Deferred Taxes (`txditc`) - Preferred Stock
- Preferred stock is calculated using a strict fallback hierarchy to maximize data retention.<br>(Redemption Value $\rightarrow$ Liquidating Value $\rightarrow$ Carrying Value)

**6.2 Construct Market Capitalization**<br>
- Closing Price (`prcc_f`) * Common Shares Outstanding (`csho`)
- Since Compustat reports shares in millions, the resulting market cap is in millions of USD.

**6.3 Construct Lagged Variables**<br>
To correctly calculate flow-over-stock ratios (like ROE) and year-over-year growth metrics, lagged variables are used. By grouping the data by ticker and using the `.shift(1)` function, the prior year's Total Assets and the Average Book Equity over the year are calculated.

**6.4 Construct the Core Factors**<br>
- **Value:** `Book-to-Market` (Book Equity / Market Cap) identifies companies trading at a discount to their net assets
- **Profitability / Quality:** `ROE` (Net Income / Avg Book Equity) and `Gross Profitability` measures how efficiently the firm generates earnings from its capital base
- **Safety / Risk:** `Leverage` (Total Debt / Total Assets) identifies firms with conservative capital structures
- **Growth:** `Asset Growth` captures the year-over-year expansion of the firm's asset base

In [ ]:
# ------------------------------------------------------------
# 6.1 Construct book equity
# ------------------------------------------------------------

# Preferred stock:
# Use redemption value first, then liquidating value, then carrying value

## 1. New column preferred_stock will start with pstkrv (redemption value)
snp500["preferred_stock"] = snp500["pstkrv"]
## 2. If pstkrv is missing, fill with pstkl (liquidating value)
if "pstkl" in snp500.columns:
    snp500["preferred_stock"] = snp500["preferred_stock"].fillna(snp500["pstkl"])
## 3. If both pstkrv and pstkl are missing, fill with pstk (carrying value)
if "pstk" in snp500.columns:
    snp500["preferred_stock"] = snp500["preferred_stock"].fillna(snp500["pstk"])
## 4. If all three are missing, fill with 0
snp500["preferred_stock"] = snp500["preferred_stock"].fillna(0)

# Deferred taxes and investment tax credit
snp500["txditc"] = snp500["txditc"].fillna(0)

# Book equity:
# Prefer SEQ; if missing, use CEQ
snp500["shareholders_equity"] = snp500["seq"].fillna(snp500["ceq"])

# Final book equity calculation:
## Book equity = shareholders' equity + deferred taxes and investment tax credit - preferred stock
snp500["book_equity"] = (
    snp500["shareholders_equity"]
    + snp500["txditc"]
    - snp500["preferred_stock"]
)

# ------------------------------------------------------------
# 6.2 Construct market capitalization
# ------------------------------------------------------------

# Compustat CSHO is usually in millions of shares
# PRCC_F is fiscal year-end price, Market cap is therefore in millions of USD
snp500["market_cap"] = snp500["prcc_f"] * snp500["csho"]

# ------------------------------------------------------------
# 6.3 Construct lagged variables
# ------------------------------------------------------------

snp500["lag_book_equity"] = snp500.groupby("ticker_yahoo")["book_equity"].shift(1)
snp500["avg_book_equity"] = (snp500["book_equity"] + snp500["lag_book_equity"]) / 2
snp500["lag_at"] = snp500.groupby("ticker_yahoo")["at"].shift(1)

# ------------------------------------------------------------
# 6.4 Generate financial signals
# ------------------------------------------------------------

# 1. Book-to-market
snp500["book_to_market"] = snp500["book_equity"] / snp500["market_cap"]
# 2. ROE
snp500["roe"] = snp500["ni"] / snp500["avg_book_equity"]
# 3. Gross profitability
snp500["gross_profitability"] = snp500["gp"] / snp500["at"]

# 4. Leverage
snp500["dlc"] = snp500["dlc"].fillna(0)
snp500["dltt"] = snp500["dltt"].fillna(0)
snp500["total_debt"] = snp500["dlc"] + snp500["dltt"]
snp500["leverage"] = snp500["total_debt"] / snp500["at"]

# 5. Asset growth
snp500["asset_growth"] = (snp500["at"] - snp500["lag_at"]) / snp500["lag_at"]

```python
# 1. Load the cleaned S&P 500 annual accounting data (ex-financials)
# Note: Ensure datadate is parsed as datetime
snp500_annual = pd.read_csv("data/snp500_cleaned_final.csv")
snp500_annual['datadate'] = pd.to_datetime(snp500_annual['datadate'], errors="coerce")

# 2. Load the cleaned Yahoo Finance monthly stock returns
monthly_returns = pd.read_csv("data/yahoo_monthly_stock_returns_clean.csv")
monthly_returns['month'] = pd.to_datetime(monthly_returns['month'])

print(f"Annual Accounting Records: {len(snp500_annual)}")
print(f"Monthly Return Records: {len(monthly_returns)}")
print("Data loaded successfully. Ready for signal generation.")
```